---
# Milestone 4: Model Training

This notebook covers:
1. **RAG Knowledge Base** — chunk, embed, and index SQLite docs, HiveQL docs, and PARROT schema
2. **Retriever** — FAISS bi-encoder top-k search
3. **Reranker** — cross-encoder rescoring to top-n
4. **RAG-augmented datasets** — context-prepended training / val / test splits
5. **Baseline training** — no RAG, establishes reference metrics
6. **Hyperparameter experiments** — LR, scheduler, beam width
7. **Optimizer experiments** — AdamW vs Adafactor
8. **Regularization experiments** — weight decay, label smoothing, dropout
9. **RAG experiment** — RAG-augmented fine-tuning vs baseline
10. **Best configuration run** — combined best settings


---
## RAG Step 1 — Build Knowledge Base

We chunk three document sources and embed them with a sentence-transformer
bi-encoder. All embeddings are stored in a FAISS flat index for cosine-similarity
retrieval at training and inference time.

| Source | Content | Why it helps |
|---|---|---|
| `sqlite_syntax.txt` | SQLite clause & function reference | Helps model understand source query intent |
| `hiveql_syntax.txt` | HiveQL dialect reference | Teaches model target syntax & dialect quirks |
| `parrot_schema.txt` | Table/column schema for PARROT + augmented data | Prevents hallucinated column/table names |


In [18]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Tuple

# ── 1a. Chunking utility ──────────────────────────────────────────────────
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 50) -> List[str]:
    """
    Split text into overlapping word-level chunks.
    chunk_size / overlap are in words (approximate token count).
    """
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(' '.join(words[start:end]))
        start += chunk_size - overlap
    return [c for c in chunks if len(c.split()) > 10]   # drop tiny tail chunks

def load_doc(path: str, label: str) -> List[Dict]:
    """Load a text file and return list of {text, source} dicts."""
    if not os.path.exists(path):
        print(f"  [WARN] {path} not found — using placeholder text for {label}.")
        # Fallback placeholder so the pipeline runs end-to-end without real docs
        placeholder = (
            f"{label} documentation placeholder. "
            "SELECT FROM WHERE GROUP BY ORDER BY HAVING JOIN LIMIT. "
            "HiveQL uses BIGINT instead of INTEGER. "
            "Use COLLECT_SET for array aggregation. "
            "LATERAL VIEW EXPLODE for array unnesting. "
            "DISTRIBUTE BY and CLUSTER BY for partitioning. "
        ) * 10
        return [{'text': c, 'source': label}
                for c in chunk_text(placeholder,
                                    train_config['chunk_size'],
                                    train_config['chunk_overlap'])]
    with open(path, 'r', encoding='utf-8') as f:
        raw = f.read()
    return [{'text': c, 'source': label}
            for c in chunk_text(raw,
                                train_config['chunk_size'],
                                train_config['chunk_overlap'])]

# ── 1b. Load all three knowledge sources ─────────────────────────────────
print("Loading knowledge sources...")
sqlite_chunks = load_doc(train_config['sqlite_doc_file'], 'sqlite_docs')
hive_chunks   = load_doc(train_config['hive_doc_file'],   'hiveql_docs')
schema_chunks = load_doc(train_config['schema_file'],     'schema')

all_chunks  = sqlite_chunks + hive_chunks + schema_chunks
chunk_texts = [c['text'] for c in all_chunks]

print(f"  SQLite doc chunks  : {len(sqlite_chunks)}")
print(f"  HiveQL doc chunks  : {len(hive_chunks)}")
print(f"  Schema chunks      : {len(schema_chunks)}")
print(f"  Total chunks       : {len(all_chunks)}")


Loading knowledge sources...
  [WARN] /content/DSAI_project/docs/sqlite_syntax.txt not found — using placeholder text for sqlite_docs.
  [WARN] /content/DSAI_project/docs/hiveql_syntax.txt not found — using placeholder text for hiveql_docs.
  [WARN] /content/DSAI_project/docs/parrot_schema.txt not found — using placeholder text for schema.
  SQLite doc chunks  : 3
  HiveQL doc chunks  : 3
  Schema chunks      : 3
  Total chunks       : 9


In [19]:
# ── 1c. Embed chunks with bi-encoder ─────────────────────────────────────
print(f"Loading bi-encoder: {train_config['retriever_model']}")
bi_encoder = SentenceTransformer(train_config['retriever_model'])

print("Embedding knowledge base chunks (this runs once offline)...")
t0 = time.time()
chunk_embeddings = bi_encoder.encode(
    chunk_texts,
    batch_size        = 64,
    show_progress_bar = True,
    normalize_embeddings = True,   # cosine via inner product
    convert_to_numpy  = True,
)
print(f"  Embedded {len(chunk_embeddings)} chunks in {time.time()-t0:.1f}s")
print(f"  Embedding dim : {chunk_embeddings.shape[1]}")

# ── 1d. Build FAISS flat index ────────────────────────────────────────────
dim   = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)          # inner product = cosine on L2-normed vecs
index.add(chunk_embeddings.astype('float32'))

print(f"  FAISS index built  : {index.ntotal} vectors, dim={dim}")

# Persist for reuse
faiss.write_index(index, 'rag_index.faiss')
import pickle
with open('rag_chunks.pkl', 'wb') as f:
    pickle.dump(all_chunks, f)
print("  Saved → rag_index.faiss, rag_chunks.pkl")


Loading bi-encoder: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding knowledge base chunks (this runs once offline)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Embedded 9 chunks in 0.6s
  Embedding dim : 384
  FAISS index built  : 9 vectors, dim=384
  Saved → rag_index.faiss, rag_chunks.pkl


---
## RAG Step 2 — Retriever & Reranker

**Bi-encoder retriever (FAISS):** Fast approximate nearest-neighbour search.
Embeds the query with the same sentence-transformer and returns the top-k
chunks by cosine similarity. Fast but coarse — may return thematically related
but imprecise chunks.

**Cross-encoder reranker:** Takes each `(query, chunk)` pair and scores them
jointly with a more powerful model. Much more accurate than bi-encoder alone
because it attends to both sequences simultaneously. We rerank top-k → top-n.


In [20]:
from sentence_transformers import CrossEncoder

print(f"Loading cross-encoder reranker: {train_config['reranker_model']}")
cross_encoder = CrossEncoder(train_config['reranker_model'], max_length=512)

def retrieve(query: str, k: int = None) -> List[Dict]:
    """Bi-encoder: embed query → FAISS top-k."""
    k = k or train_config['rag_top_k']
    q_emb = bi_encoder.encode([query], normalize_embeddings=True,
                               convert_to_numpy=True)
    scores, indices = index.search(q_emb.astype('float32'), k)
    return [{'text'  : all_chunks[i]['text'],
             'source': all_chunks[i]['source'],
             'score' : float(scores[0][j])}
            for j, i in enumerate(indices[0])]

def rerank(query: str, candidates: List[Dict], n: int = None) -> List[Dict]:
    """Cross-encoder: rescore candidates and keep top-n."""
    n = n or train_config['rag_top_n']
    if not candidates:
        return []
    pairs  = [(query, c['text']) for c in candidates]
    scores = cross_encoder.predict(pairs, show_progress_bar=False)
    for c, s in zip(candidates, scores):
        c['rerank_score'] = float(s)
    return sorted(candidates, key=lambda x: x['rerank_score'], reverse=True)[:n]

def retrieve_and_rerank(query: str) -> List[Dict]:
    """Full RAG retrieval pipeline: bi-encoder → cross-encoder → top-n chunks."""
    candidates = retrieve(query, k=train_config['rag_top_k'])
    return rerank(query, candidates, n=train_config['rag_top_n'])

# ── Quick sanity check ────────────────────────────────────────────────────
test_q = "SELECT COUNT(*) FROM orders WHERE status = 'pending'"
results = retrieve_and_rerank(test_q)
print(f"\nRetrieval test: '{test_q[:60]}...'")
for i, r in enumerate(results, 1):
    print(f"  [{i}] source={r['source']}  rerank={r['rerank_score']:.3f}")
    print(f"       {r['text'][:100]}...")


Loading cross-encoder reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]


Retrieval test: 'SELECT COUNT(*) FROM orders WHERE status = 'pending'...'
  [1] source=hiveql_docs  rerank=-7.469
       placeholder. SELECT FROM WHERE GROUP BY ORDER BY HAVING JOIN LIMIT. HiveQL uses BIGINT instead of IN...
  [2] source=hiveql_docs  rerank=-7.938
       hiveql_docs documentation placeholder. SELECT FROM WHERE GROUP BY ORDER BY HAVING JOIN LIMIT. HiveQL...
  [3] source=hiveql_docs  rerank=-9.162
       FROM WHERE GROUP BY ORDER BY HAVING JOIN LIMIT. HiveQL uses BIGINT instead of INTEGER. Use COLLECT_S...


---
## RAG Step 3 — Context Assembly & RAG-Augmented Datasets

We build two parallel dataset variants:

- **No-RAG** (already built above as `tokenised_train/val/test`) — baseline
- **RAG-augmented** — each input is prepended with a `<context>` block containing
  the top-3 retrieved chunks, separated by `<sep>` tokens

Input format for RAG variant:
```
<sqlite> SELECT ... FROM orders WHERE ...
<context> [HiveQL: COUNT(*) returns BIGINT] <sep> [Schema: orders(...)] <sep> [...]
```

Both variants are tokenised to `max_ip_ln=512` so they share the same model.


In [21]:
def build_context_string(chunks: List[Dict]) -> str:
    """Concatenate retrieved chunk texts with <sep> separators."""
    parts = [c['text'] for c in chunks]
    return ' <sep> '.join(parts)

def preprocess_rag(batch):
    """
    RAG-augmented preprocessing.
    For each SQLite query we retrieve context at tokenisation time and
    prepend it to the input sequence.
    """
    augmented_inputs = []
    for q in batch['sqlite']:
        chunks  = retrieve_and_rerank(q)
        context = build_context_string(chunks)
        aug_input = (train_config['src_pfx'] + q +
                     ' ' + train_config['ctx_pfx'] + ' ' + context)
        augmented_inputs.append(aug_input)

    targets = [train_config['tgt_pfx'] + q for q in batch['hive']]

    model_inputs = tokenizer(
        augmented_inputs,
        max_length  = train_config['max_ip_ln'],
        padding     = 'max_length',
        truncation  = True,
    )
    labels = tokenizer(
        targets,
        max_length  = train_config['max_op_ln'],
        padding     = 'max_length',
        truncation  = True,
    )
    model_inputs['labels'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in lbl]
        for lbl in labels['input_ids']
    ]
    return model_inputs

# ── Build RAG-augmented splits ────────────────────────────────────────────
# NOTE: retrieval runs once per sample — may take a few minutes on large splits.
# We use batch_size=1 to avoid RAM spikes from simultaneous FAISS lookups.
print("Building RAG-augmented tokenised datasets...")
print("  (retrieval runs per sample — this is the longest setup step)")

t0 = time.time()
rag_tokenised_train = train_ds.map(
    preprocess_rag, batched=True, batch_size=32,
    remove_columns=['sqlite','hive'],
    desc="RAG train"
)
rag_tokenised_val = val_ds.map(
    preprocess_rag, batched=True, batch_size=32,
    remove_columns=['sqlite','hive'],
    desc="RAG val"
)
rag_tokenised_test = test_ds.map(
    preprocess_rag, batched=True, batch_size=32,
    remove_columns=['sqlite','hive'],
    desc="RAG test"
)
print(f"  Done in {(time.time()-t0)/60:.1f} min")
print(f"  RAG train : {len(rag_tokenised_train):,}")
print(f"  RAG val   : {len(rag_tokenised_val):,}")
print(f"  RAG test  : {len(rag_tokenised_test):,}")

# ── Verify a sample RAG input ─────────────────────────────────────────────
sample_q = train_df['sqlite'].iloc[0]
sample_chunks = retrieve_and_rerank(sample_q)
sample_ctx    = build_context_string(sample_chunks)
sample_aug    = train_config['src_pfx'] + sample_q + ' ' + train_config['ctx_pfx'] + ' ' + sample_ctx
n_toks = tokenizer(sample_aug, return_length=True)['length'][0]
print(f"\nSample RAG input ({n_toks} tokens):")
print(sample_aug[:300], '...')


Building RAG-augmented tokenised datasets...
  (retrieval runs per sample — this is the longest setup step)


Parameter 'function'=<function preprocess_rag at 0x7b56ecbeafc0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


RAG train:   0%|          | 0/3064 [00:00<?, ? examples/s]

RAG val:   0%|          | 0/963 [00:00<?, ? examples/s]

RAG test:   0%|          | 0/888 [00:00<?, ? examples/s]

  Done in 3.4 min
  RAG train : 3,064
  RAG val   : 963
  RAG test  : 888

Sample RAG input (670 tokens):
<sqlite>SELECT DISTINCT COUNT( PAPERalias0.PAPERID ) FROM AUTHOR AS AUTHORalias0 , PAPER AS PAPERalias0 , WRITES AS WRITESalias0 WHERE AUTHORalias0.AUTHORNAME = "authorname0" AND PAPERalias0.YEAR == YEAR(CURDATE()) - misc0 AND WRITESalias0.AUTHORID = AUTHORalias0.AUTHORID AND WRITESalias0.PAPERID =  ...


---
## Experiment Tracker

In [22]:
import json, copy, time
from dataclasses import dataclass, field, asdict
from typing import Optional

@dataclass
class ExperimentResult:
    name            : str
    description     : str
    use_rag         : bool    = False
    learning_rate   : float   = 3e-4
    batch_size      : int     = 8
    warmup_ratio    : float   = 0.06
    weight_decay    : float   = 0.01
    lr_scheduler    : str     = 'cosine'
    optimizer       : str     = 'adamw_torch'
    num_beams       : int     = 4
    label_smoothing : float   = 0.0
    dropout         : float   = 0.1
    val_exact_match : Optional[float] = None
    val_bleu        : Optional[float] = None
    val_parse_valid : Optional[float] = None
    train_time_min  : Optional[float] = None

experiment_log: list = []

def print_results_table():
    rows = [{
        'Experiment'  : r.name,
        'RAG'         : '✓' if r.use_rag else '✗',
        'LR'          : r.learning_rate,
        'Optim'       : r.optimizer,
        'WD'          : r.weight_decay,
        'LblSmooth'   : r.label_smoothing,
        'Beams'       : r.num_beams,
        'ExactMatch%' : r.val_exact_match,
        'BLEU'        : r.val_bleu,
        'ParseValid%' : r.val_parse_valid,
        'Time(min)'   : r.train_time_min,
    } for r in experiment_log]
    df = pd.DataFrame(rows)
    print("\n" + "="*95)
    print("EXPERIMENT RESULTS SUMMARY")
    print("="*95)
    print(df.to_string(index=False))
    print("="*95 + "\n")

print("Experiment tracker ready.")


Experiment tracker ready.


## Training Helper Function

In [49]:
from transformers import (
    T5ForConditionalGeneration, Seq2SeqTrainingArguments,
    Seq2SeqTrainer, DataCollatorForSeq2Seq, EarlyStoppingCallback,
)
from transformers.optimization import Adafactor, AdafactorSchedule
from peft import LoraConfig, get_peft_model, TaskType


def build_and_train(
    exp_name        : str,
    description     : str,
    use_rag         : bool  = False,
    learning_rate   : float = 3e-4,
    batch_size      : int   = 8,
    warmup_ratio    : float = 0.06,
    weight_decay    : float = 0.01,
    lr_scheduler    : str   = 'cosine',
    optimizer_name  : str   = 'adamw_torch',
    num_beams       : int   = 4,
    label_smoothing : float = 0.0,
    dropout_rate    : float = 0.1,
    num_epochs      : int   = 5,
    grad_accum      : int   = 4,
    use_lora        : bool  = True,   # 👈 NEW FLAG
) -> ExperimentResult:

    set_seed(train_config['seed'])

    _model = T5ForConditionalGeneration.from_pretrained(train_config['model_name'])
    _model.resize_token_embeddings(len(tokenizer))

    if dropout_rate != 0.1:
        _model.config.dropout_rate = dropout_rate

    # ✅ Enable gradient checkpointing (huge memory saver)
    _model.gradient_checkpointing_enable()

    # ✅ Apply LoRA
    if use_lora:
        lora_config = LoraConfig(
            r=8,
            lora_alpha=16,
            target_modules=["q", "v"],  # T5 attention projections
            lora_dropout=0.1,
            bias="none",
            task_type=TaskType.SEQ_2_SEQ_LM,
        )

        _model = get_peft_model(_model, lora_config)
        _model.print_trainable_parameters()

    _model = _model.to(device)

    # Select dataset variant
    _train_ds = rag_tokenised_train if use_rag else tokenised_train
    _val_ds   = rag_tokenised_val   if use_rag else tokenised_val

    ckpt_dir = f"checkpoints/{exp_name.replace(' ','_')}"
    os.makedirs(ckpt_dir, exist_ok=True)

    _collator = DataCollatorForSeq2Seq(
        tokenizer, model=_model, label_pad_token_id=-100, pad_to_multiple_of=8
    )

    optimizers = (None, None)

    if optimizer_name == 'adafactor':
        _opt = Adafactor(
            _model.parameters(),
            scale_parameter=True,
            relative_step=True,
            warmup_init=True,
            lr=None
        )
        _sch = AdafactorSchedule(_opt)
        optimizers = (_opt, _sch)
        _lr, _sched, _warmup = 1e-3, 'constant', 0.0
    else:
        _lr, _sched, _warmup = learning_rate, lr_scheduler, warmup_ratio

    args = Seq2SeqTrainingArguments(
        output_dir                  = ckpt_dir,
        num_train_epochs            = num_epochs,
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = batch_size,
        gradient_accumulation_steps = grad_accum,
        learning_rate               = _lr,
        warmup_ratio                = _warmup,
        weight_decay                = weight_decay,
        lr_scheduler_type           = _sched,
        label_smoothing_factor      = label_smoothing,
        fp16                        = train_config['fp16'],
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        metric_for_best_model       = train_config['metric_for_best'],
        greater_is_better           = True,
        save_total_limit            = 2,
        predict_with_generate       = True,
        generation_num_beams        = num_beams,
        generation_max_length       = train_config['max_new_tokens'],
        logging_dir                 = f'./logs/{exp_name}',
        logging_steps               = 50,
        seed                        = train_config['seed'],
        report_to                   = 'none',
    )

    trainer = Seq2SeqTrainer(
        model           = _model,
        args            = args,
        train_dataset   = _train_ds,
        eval_dataset    = _val_ds,
        data_collator   = _collator,
        compute_metrics = compute_metrics_trainer,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
        optimizers      = optimizers,
    )

    rag_tag = '[RAG]' if use_rag else '[no-RAG]'
    lora_tag = '[LoRA]' if use_lora else '[FullFT]'

    print(f"\n{'='*60}")
    print(f"  {rag_tag} {lora_tag} EXPERIMENT: {exp_name}")
    print(f"  {description}")
    print(f"{'='*60}")

    t0 = time.time()
    trainer.train()

    log_history = trainer.state.log_history
    eval_logs   = [l for l in log_history if 'eval_exact_match' in l]
    last_eval   = eval_logs[-1] if eval_logs else {}

    result = ExperimentResult(
        name            = exp_name,
        description     = description,
        use_rag         = use_rag,
        learning_rate   = learning_rate,
        batch_size      = batch_size,
        warmup_ratio    = warmup_ratio,
        weight_decay    = weight_decay,
        lr_scheduler    = lr_scheduler,
        optimizer       = optimizer_name,
        num_beams       = num_beams,
        label_smoothing = label_smoothing,
        dropout         = dropout_rate,
        val_exact_match = last_eval.get('eval_exact_match'),
        val_bleu        = last_eval.get('eval_bleu'),
        val_parse_valid = last_eval.get('eval_parse_validity'),
        train_time_min  = round((time.time() - t0) / 60, 1),
    )

    experiment_log.append(result)
    print_results_table()

    del _model, trainer
    torch.cuda.empty_cache()

    return result


print("build_and_train() with LoRA ready.")

build_and_train() with LoRA ready.


---
## Experiment 1 — Baseline (no RAG)

Standard T5-base fine-tuning with Milestone 3 config. No retrieval context.
This is the reference point that all subsequent experiments are compared against.


In [50]:
baseline_result = build_and_train(
    exp_name      = "Baseline",
    description   = "T5-base, AdamW lr=3e-4, cosine, WD=0.01, beams=4, no RAG",
    use_rag       = False,
    learning_rate = 1e-5,
    num_epochs    = 5,
)


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

trainable params: 884,736 || all params: 223,769,856 || trainable%: 0.3954


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



  [no-RAG] [LoRA] EXPERIMENT: Baseline
  T5-base, AdamW lr=3e-4, cosine, WD=0.01, beams=4, no RAG


Epoch,Training Loss,Validation Loss,Exact Match,Bleu,Parse Validity
1,No log,5.680256,0.000000,49.190000,3.220000
2,46.648403,5.583907,0.000000,49.590000,3.120000
3,44.960542,5.489214,0.000000,49.800000,3.010000
4,45.463418,5.441835,0.000000,50.300000,2.910000


TypeError: ExperimentResult.__init__() got an unexpected keyword argument 'warmup_steps'

---
## Experiment 2 — Learning Rate Sweep

Tests lr ∈ {1e-4, 5e-4} against the baseline 3e-4. No RAG — isolates LR effect.


In [51]:
for lr, tag in [(1e-4, "LR-1e4"), (5e-4, "LR-5e4")]:
    build_and_train(
        exp_name      = tag,
        description   = f"LR ablation lr={lr}, no RAG",
        use_rag       = False,
        learning_rate = lr,
        num_epochs    = 8,
    )


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

trainable params: 884,736 || all params: 223,769,856 || trainable%: 0.3954


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



  [no-RAG] [LoRA] EXPERIMENT: LR-1e4
  LR ablation lr=0.0001, no RAG


KeyboardInterrupt: 

---
## Experiment 3 — LR Scheduler

Compares cosine (baseline) vs linear vs constant_with_warmup.
Uses best LR from Exp 2 — update `best_lr` before running.


In [ ]:
best_lr = 3e-4   # update if Exp 2 found a better LR
for sched in ['linear', 'constant_with_warmup']:
    build_and_train(
        exp_name      = f"Sched-{sched}",
        description   = f"Scheduler={sched}, lr={best_lr}, no RAG",
        use_rag       = False,
        learning_rate = best_lr,
        lr_scheduler  = sched,
        num_epochs    = 8,
    )


---
## Experiment 4 — Beam Search Width (inference only)

Reloads the best no-RAG checkpoint and evaluates with beam widths 2, 4, 6, 8.
No retraining — pure inference experiment.


In [28]:
def eval_beam_width(beam_width: int, ckpt_dir: str = "checkpoints/Baseline"):
    set_seed(train_config['seed'])
    _model = T5ForConditionalGeneration.from_pretrained(ckpt_dir)
    _model = _model.to(device)
    _col = DataCollatorForSeq2Seq(tokenizer, model=_model,
                                   label_pad_token_id=-100, pad_to_multiple_of=8)
    _args = Seq2SeqTrainingArguments(
        output_dir                 = f"checkpoints/beam_{beam_width}",
        per_device_eval_batch_size = 8,
        predict_with_generate      = True,
        generation_num_beams       = beam_width,
        generation_max_length      = train_config['max_new_tokens'],
        fp16                       = train_config['fp16'],
        report_to                  = 'none',
    )
    t = Seq2SeqTrainer(model=_model, args=_args, eval_dataset=tokenised_val,
                       data_collator=_col, compute_metrics=compute_metrics_trainer)
    t0 = time.time()
    metrics = t.evaluate()
    result = ExperimentResult(
        name=f"Beam-{beam_width}",
        description=f"Beam width={beam_width} on baseline checkpoint",
        num_beams=beam_width,
        val_exact_match=metrics.get('eval_exact_match'),
        val_bleu=metrics.get('eval_bleu'),
        val_parse_valid=metrics.get('eval_parse_validity'),
        train_time_min=round((time.time()-t0)/60, 2),
    )
    experiment_log.append(result)
    print(f"  Beams={beam_width}: EM={result.val_exact_match}  BLEU={result.val_bleu}")
    del _model, t
    torch.cuda.empty_cache()

print("eval_beam_width() defined.")
print("Run after Baseline training:")
print("  for b in [2, 6, 8]: eval_beam_width(b)")
# Uncomment to execute:
# for b in [2, 6, 8]:
#     eval_beam_width(b)


eval_beam_width() defined.
Run after Baseline training:
  for b in [2, 6, 8]: eval_beam_width(b)


---
## Experiment 5 — Optimizer: AdamW vs Adafactor

Adafactor is the original T5 optimizer — memory-efficient, self-schedules LR.


In [29]:
build_and_train(
    exp_name       = "Adafactor",
    description    = "Adafactor optimizer, relative_step=True, no RAG",
    use_rag        = False,
    optimizer_name = 'adafactor',
    num_epochs     = 8,
)


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 96.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 11.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.06 GiB is allocated by PyTorch, and 296.09 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

---
## Experiment 6 — Regularization

### 6a Weight Decay  |  6b Label Smoothing  |  6c Dropout


In [30]:
# 6a. Weight decay
for wd in [0.0, 0.05, 0.1]:
    build_and_train(
        exp_name      = f"WD-{wd}",
        description   = f"Weight decay={wd}, no RAG",
        use_rag       = False,
        weight_decay  = wd,
        num_epochs    = 8,
    )


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 96.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 11.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.06 GiB is allocated by PyTorch, and 296.09 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [31]:
# 6b. Label smoothing
for ls in [0.05, 0.1]:
    build_and_train(
        exp_name        = f"LabelSmooth-{ls}",
        description     = f"Label smoothing eps={ls}, no RAG",
        use_rag         = False,
        label_smoothing = ls,
        num_epochs      = 8,
    )


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 96.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 11.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.06 GiB is allocated by PyTorch, and 296.09 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [32]:
# 6c. Dropout
for dr in [0.0, 0.2]:
    build_and_train(
        exp_name     = f"Dropout-{dr}",
        description  = f"Dropout rate={dr}, no RAG",
        use_rag      = False,
        dropout_rate = dr,
        num_epochs   = 8,
    )


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 96.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 11.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.06 GiB is allocated by PyTorch, and 296.09 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

---
## Experiment 7 — RAG-Augmented Fine-Tuning

This is the key new experiment. We fine-tune T5-base with **RAG context prepended**
to every training input. The model learns to:
- Use retrieved HiveQL syntax hints when generating output
- Ground column/table names in the retrieved schema
- Leverage dialect difference notes between SQLite and HiveQL

The RAG pipeline uses:
- **Bi-encoder**: `sentence-transformers/all-MiniLM-L6-v2` retrieves top-5 chunks
- **Cross-encoder reranker**: `cross-encoder/ms-marco-MiniLM-L-6-v2` rescores → top-3
- **Context format**: `<sqlite> {query} <context> {chunk1} <sep> {chunk2} <sep> {chunk3}`

Input sequence length is extended to 512 tokens (vs 256 for baseline).
Batch size is halved to 8 with grad_accum=4 to maintain effective batch=32.


In [ ]:
build_and_train(
    exp_name      = "RAG-Augmented",
    description   = "T5-base + RAG context (top-3 reranked), lr=3e-4, cosine, WD=0.01",
    use_rag       = True,
    learning_rate = 5e-5,
    batch_size    = 4,
    grad_accum    = 4,
    num_epochs    = 8,
)


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

trainable params: 884,736 || all params: 223,769,856 || trainable%: 0.3954


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



  [RAG] [LoRA] EXPERIMENT: RAG-Augmented
  T5-base + RAG context (top-3 reranked), lr=3e-4, cosine, WD=0.01


Epoch,Training Loss,Validation Loss,Exact Match,Bleu,Parse Validity
1,45.589282,2.646047,0.000000,34.380000,1.770000
2,13.027316,0.679998,0.000000,60.850000,2.280000
3,5.703096,0.407448,0.830000,65.110000,4.470000
4,3.953564,0.322279,16.610000,81.470000,0.000000
5,3.181461,0.281882,19.830000,85.230000,0.100000
6,2.891814,0.264488,21.390000,85.710000,0.100000
7,2.782863,0.257821,22.120000,85.780000,0.100000


---
## Experiment 8 — RAG Ablation: Retriever Only vs Retriever + Reranker

We compare two RAG variants to measure the reranker's contribution:
- **RAG-NoRerank**: uses top-3 by bi-encoder score only (no cross-encoder)
- **RAG-Reranked**: bi-encoder top-5 → cross-encoder → top-3 (Experiment 7)


In [34]:
# Patch retrieve_and_rerank to skip reranker for this ablation
def preprocess_rag_no_rerank(batch):
    """RAG without cross-encoder reranking — bi-encoder top-n directly."""
    augmented_inputs = []
    for q in batch['sqlite']:
        candidates = retrieve(q, k=train_config['rag_top_n'])   # no reranking
        context    = build_context_string(candidates)
        augmented_inputs.append(
            train_config['src_pfx'] + q + ' ' +
            train_config['ctx_pfx'] + ' ' + context
        )
    targets = [train_config['tgt_pfx'] + q for q in batch['hive']]
    model_inputs = tokenizer(augmented_inputs,
                              max_length=train_config['max_ip_ln'],
                              padding='max_length', truncation=True)
    labels = tokenizer(targets, max_length=train_config['max_op_ln'],
                       padding='max_length', truncation=True)
    model_inputs['labels'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in lbl]
        for lbl in labels['input_ids']
    ]
    return model_inputs

print("Building no-rerank RAG datasets...")
rag_nr_train = train_ds.map(preprocess_rag_no_rerank, batched=True,
                             batch_size=32, remove_columns=['sqlite','hive'],
                             desc="RAG-NoRerank train")
rag_nr_val   = val_ds.map(preprocess_rag_no_rerank, batched=True,
                           batch_size=32, remove_columns=['sqlite','hive'],
                           desc="RAG-NoRerank val")
print("Done.")

# ── Temporarily override dataset in build_and_train by monkey-patching ─────
_orig_rag_train = rag_tokenised_train
_orig_rag_val   = rag_tokenised_val
rag_tokenised_train = rag_nr_train
rag_tokenised_val   = rag_nr_val

build_and_train(
    exp_name  = "RAG-NoRerank",
    description = "RAG top-3 bi-encoder only (no cross-encoder reranker)",
    use_rag   = True,
    num_epochs = 8,
)

rag_tokenised_train = _orig_rag_train
rag_tokenised_val   = _orig_rag_val
print("Restored reranked RAG datasets.")


Building no-rerank RAG datasets...


RAG-NoRerank train:   0%|          | 0/3064 [00:00<?, ? examples/s]

RAG-NoRerank val:   0%|          | 0/963 [00:00<?, ? examples/s]

Done.


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 96.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 11.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.06 GiB is allocated by PyTorch, and 296.09 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

---
## Experiment 9 — Best Configuration (Combined)

Combine the best hyperparameters found in Experiments 2-6 with RAG augmentation.
Update the variables below based on your results table before running.


In [ ]:
# ── Update these from your results table ─────────────────────────────────
best_lr        = 3e-4
best_scheduler = 'cosine'
best_wd        = 0.01
best_ls        = 0.05
best_dr        = 0.1
best_beams     = 4

build_and_train(
    exp_name        = "BestConfig-RAG",
    description     = "Best hyperparams + RAG augmentation — full 10-epoch run",
    use_rag         = True,
    learning_rate   = best_lr,
    lr_scheduler    = best_scheduler,
    weight_decay    = best_wd,
    label_smoothing = best_ls,
    dropout_rate    = best_dr,
    num_beams       = best_beams,
    num_epochs      = 10,
)


---
## Final Results Summary & Test Set Evaluation

In [35]:
print_results_table()

valid = [r for r in experiment_log if r.val_exact_match is not None]
if valid:
    best = max(valid, key=lambda r: r.val_exact_match)
    print(f"Best experiment : {best.name}  ({'RAG' if best.use_rag else 'no-RAG'})")
    print(f"  Exact Match   : {best.val_exact_match}%")
    print(f"  BLEU          : {best.val_bleu}")
    print(f"  Parse Validity: {best.val_parse_valid}%")



EXPERIMENT RESULTS SUMMARY
Empty DataFrame
Columns: []
Index: []



In [36]:
from dataclasses import asdict
log_df = pd.DataFrame([asdict(r) for r in experiment_log])
log_df.to_csv('milestone4_experiment_log.csv', index=False)
print("Saved → milestone4_experiment_log.csv")
print(log_df[['name','use_rag','val_exact_match','val_bleu','val_parse_valid','train_time_min']])


Saved → milestone4_experiment_log.csv


KeyError: "None of [Index(['name', 'use_rag', 'val_exact_match', 'val_bleu', 'val_parse_valid',\n       'train_time_min'],\n      dtype='object')] are in the [columns]"

---
## Test Set Evaluation (Best Model)

Load the best checkpoint and run on the held-out test set.
Update `BEST_EXP_NAME` to the experiment name that won.


In [37]:
BEST_EXP_NAME = "BestConfig-RAG"   # update if different
BEST_CKPT_DIR = f"checkpoints/{BEST_EXP_NAME.replace(' ','_')}"
BEST_USE_RAG  = True                # set False if best was a no-RAG run

_model = T5ForConditionalGeneration.from_pretrained(BEST_CKPT_DIR)
_model = _model.to(device)
_col   = DataCollatorForSeq2Seq(tokenizer, model=_model,
                                 label_pad_token_id=-100, pad_to_multiple_of=8)
_test_ds = rag_tokenised_test if BEST_USE_RAG else tokenised_test

test_args = Seq2SeqTrainingArguments(
    output_dir                 = "checkpoints/test_eval",
    per_device_eval_batch_size = 8,
    predict_with_generate      = True,
    generation_num_beams       = best_beams,
    generation_max_length      = train_config['max_new_tokens'],
    fp16                       = train_config['fp16'],
    report_to                  = 'none',
)
test_trainer = Seq2SeqTrainer(
    model=_model, args=test_args, data_collator=_col,
    compute_metrics=compute_metrics_trainer,
)
test_results = test_trainer.predict(_test_ds)
pred_ids  = np.clip(test_results.predictions, 0, len(tokenizer)-1)
label_ids = np.clip(
    np.where(test_results.label_ids != -100,
             test_results.label_ids, tokenizer.pad_token_id),
    0, len(tokenizer)-1
)
decoded_preds  = tokenizer.batch_decode(pred_ids.tolist(),  skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(label_ids.tolist(), skip_special_tokens=True)
final = compute_metrics(decoded_preds, decoded_labels)

print("\n" + "="*50)
print("TEST SET RESULTS")
print("="*50)
for k, v in final.items():
    print(f"  {k:20s}: {v}")
print("="*50)


OSError: checkpoints/BestConfig-RAG is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`